In [16]:
import os
import random
import shutil
import pathlib

import numpy as np

os.environ["KERAS_BACKEND"] = "jax"

import keras

### Using GloVe vectors

Just as with pretrained models, we can also obtain vectors that have been trained on very large amounts of text.

Here, we will download [GloVe vectors](https://nlp.stanford.edu/projects/glove/).

(The same method applies to Word2Vec or any other embedding technique.)

#### Download & process the vectors

In [2]:
# I have my file in a folder called 'glove'
GLOVE_DIR = pathlib.Path("glove")

if not GLOVE_DIR.exists():
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip -d glove # unzip to a directory called "glove"
    !rm glove.6B.zip                # remove the zip file

PATH_TO_GLOVE_FILE =  GLOVE_DIR / "glove.6B.100d.txt"

In [3]:
# another Jupyter magic: use $python_variable in bash commands
!head -n 1 $PATH_TO_GLOVE_FILE
# ↓ the word "the" followed by its coordinates in a 100-dimensional space

the -0.038194 -0.24487 0.72812 -0.39961 0.083172 0.043953 -0.39141 0.3344 -0.57545 0.087459 0.28787 -0.06731 0.30906 -0.26384 -0.13231 -0.20757 0.33395 -0.33848 -0.31743 -0.48336 0.1464 -0.37304 0.34577 0.052041 0.44946 -0.46971 0.02628 -0.54155 -0.15518 -0.14107 -0.039722 0.28277 0.14393 0.23464 -0.31021 0.086173 0.20397 0.52624 0.17164 -0.082378 -0.71787 -0.41531 0.20335 -0.12763 0.41367 0.55187 0.57908 -0.33477 -0.36559 -0.54857 -0.062892 0.26584 0.30205 0.99775 -0.80481 -3.0243 0.01254 -0.36942 2.2167 0.72201 -0.24978 0.92136 0.034514 0.46745 1.1079 -0.19358 -0.074575 0.23353 -0.052062 -0.22044 0.057162 -0.15806 -0.30798 -0.41625 0.37972 0.15006 -0.53212 -0.2055 -1.2526 0.071624 0.70565 0.49744 -0.42063 0.26148 -1.538 -0.30223 -0.073438 -0.28312 0.37104 -0.25217 0.016215 -0.017099 -0.38984 0.87424 -0.72569 -0.51058 -0.52028 -0.1459 0.8278 0.27062


In [4]:
# parsing the GloVe word-embeddings file

# our dictionary: {'word': np.array([...coordinates..])}
embeddings_index = {}
glove_words = []
glove_matrix = []
with open(PATH_TO_GLOVE_FILE) as f:
    for line in f:
        # split: word | coordinates
        word, coefs = line.split(maxsplit=1)
        # load string floats into numpy, space-separated
        coefs = np.fromstring(coefs, "float", sep=" ")
        # save into dictionary
        embeddings_index[word] = coefs
        glove_words.append(word)
        glove_matrix.append(coefs)

print(f"Found {len(embeddings_index):,} word vectors of dim {glove_matrix[0].shape[0]}.")

glove_matrix = np.stack(glove_matrix)
# L2 normalize everything (for easier computation later
glove_matrix = glove_matrix / np.linalg.norm(glove_matrix, axis=1, keepdims=True)
glove_word_to_idx = {w: i for i, w in enumerate(glove_words)}

Found 400,000 word vectors of dim 100.


The way you would test embeddings you trained yourself is this:

```python
cbow_embedding = keras.models.load_model(MODELS_DIR/ "cbow_embeddings.keras")

# if you still have the defined model in memory, you can use:
# cbow_matrix = cbow_embedding.embeddings.numpy()

# when reloading, use the weights
cbow_matrix = cbow_embedding.weights[0].numpy()

# L2 normalize everything (for easier computation later
cbow_matrix = cbow_matrix / np.linalg.norm(cbow_matrix, axis=1, keepdims=True)

cbow_word_to_idx = {str(w): i for i, w in enumerate(tokenize_no_padding.get_vocabulary())}
```

Note, however, that at least in my experiments, the vectors from the model above don't yield anything meaningful! It might be a case for
1. Training for longer;
2. Increasing the `hidden_dim`;
3. Train on a larger / more diverse amount of text.

#### Word vector arithmetic

In [5]:
def closest(word, words, word_to_idx, embedding_matrix, k=10):
    """
    Find the closest word vectors to a given word
    """
    idx = word_to_idx[word]
    # already normalized
    vec = embedding_matrix[idx]

    # cosine similarity (just the dot product/matmul + normalisation (already done))
    sims = embedding_matrix @ vec
    # partial sort (much faster)
    best = np.argpartition(-sims, k+1)[:k+1]
    # exact order inside top-k
    best = best[np.argsort(-sims[best])]
    return [words[i] for i in best if i != idx][:k]

In [12]:
closest("berlin", glove_words, glove_word_to_idx, glove_matrix)

['munich',
 'vienna',
 'hamburg',
 'warsaw',
 'bonn',
 'germany',
 'prague',
 'dresden',
 'frankfurt',
 'cologne']

In [7]:
closest("cat", glove_words, glove_word_to_idx, glove_matrix)

['dog',
 'rabbit',
 'cats',
 'monkey',
 'pet',
 'dogs',
 'mouse',
 'puppy',
 'rat',
 'spider']

In [8]:
def analogy(w1, w2, w3, words, word_to_idx, embedding_matrix, k=10):
    """
    Performs the word vector arithmetic: "w1 is to w2 what w3 is to...", by doing
    (vec2 - vec1) + vec3
    Returns the closest vectors.
    """
    vec = (
        embedding_matrix[word_to_idx[w2]]
        - embedding_matrix[word_to_idx[w1]]
        + embedding_matrix[word_to_idx[w3]]
    )
    # normalize analogy vector
    vec = vec / np.linalg.norm(vec)

    # cosine similarity (just the dot product/matmul + normalisation (already done))
    sims = embedding_matrix @ vec
    # partial sort (much faster)
    best = np.argpartition(-sims, k+3)[:k+3]
    # exact order inside top-k
    best = best[np.argsort(-sims[best])]

    excluded = { word_to_idx[w1], word_to_idx[w2], word_to_idx[w3] }
    return [words[i] for i in best if i not in excluded][:k]

In [9]:
# Tokyo is to Japan what Berlin is to...
analogy("tokyo", "japan", "berlin", glove_words, glove_word_to_idx, glove_matrix)

['germany',
 'austria',
 'denmark',
 'german',
 'poland',
 'munich',
 'europe',
 'italy',
 'finland',
 'hungary']

In [10]:
# Show is to showed what sing is to...
analogy("show", "showed", "sing", glove_words, glove_word_to_idx, glove_matrix)
# (try with 'shows', works as well)

['sang',
 'sung',
 'hung',
 'singing',
 'sings',
 'listened',
 'prayed',
 'cried',
 'hymns',
 'heard']

In [11]:
# Wolf is to dog what tiger is to...
analogy("wolf", "dog", "tiger", glove_words, glove_word_to_idx, glove_matrix)
# (funnily enough, this sort of fails in the other direction)

['cat',
 'horse',
 'cats',
 'pet',
 'dogs',
 'stray',
 'elephant',
 'crocodile',
 'golf',
 'golfer']

### Extra: Import GloVe vectors into an `Embedding` layer

#### Loading IMDB

In [13]:
DATASET_DIR = pathlib.Path("aclImdb")

if not DATASET_DIR.exists():
    !curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
    !tar -xf aclImdb_v1.tar.gz # this untars the archive to a folder called aclImdb
    !rm -r aclImdb/train/unsup

MODELS_DIR = pathlib.Path("models")
MODELS_DIR.mkdir(exist_ok=True)

# to read a review:
# !cat aclImdb/train/pos/4077_10.txt

In [17]:
# code to split the data into train/val folders

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
TEST_DIR = DATASET_DIR / "test"
for category in ("neg", "pos"):
    if not os.path.isdir(VAL_DIR / category):    # do this only once
        os.makedirs(VAL_DIR / category)          # make 'neg'/'pos' dir in validation
        files = os.listdir(TRAIN_DIR / category) # list files in 'train'
        random.Random(1337).shuffle(files)       # shuffle using a seed
        num_val_samples = int(0.2 * len(files))  # 2% of our samples for validation
        val_files = files[-num_val_samples:]
        for fname in val_files:                  # move our files
            shutil.move(TRAIN_DIR / category / fname,
                        VAL_DIR / category / fname)

In [19]:
BATCH_SIZE = 32

# each of these iterables returns tuples containing two tensors:
# samples, shape: (batch_size, sample_shape) ← our texts
# targets, shape: (batch_size,)              ← 0 or 1
train_ds = keras.utils.text_dataset_from_directory(
    TRAIN_DIR, batch_size=BATCH_SIZE
)
val_ds = keras.utils.text_dataset_from_directory(
    VAL_DIR, batch_size=BATCH_SIZE
)
test_ds = keras.utils.text_dataset_from_directory(
    TEST_DIR, batch_size=BATCH_SIZE
)

text_only_train_ds = train_ds.map(lambda x, y: x)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [20]:
max_length = 600
max_tokens = 8000

text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(text_only_train_ds)

sequence_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
sequence_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
sequence_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [21]:
embedding_dim = 100

# we reuse the same TextVectorization object as earlier, turning our sentences into integers
# max_length: 600, max_tokens: 8000
vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

# preparing the GloVe word-embeddings matrix
embedding_matrix = np.zeros((max_tokens, embedding_dim))     # create a matrix (max_tokens, embedding_dim)
for word, i in word_index.items():                           # looping through our vocab
    if i < max_tokens:                                       # don't try and retrieve beyond max_tokens
        embedding_vector = embeddings_index.get(word)        # try and get the vector associated with the word
    if embedding_vector is not None:                         # if the vector exists
        embedding_matrix[i] = embedding_vector               # assign it to our matrix

In [22]:
embedding_layer = keras.layers.Embedding(
    max_tokens,
    embedding_dim,        # using our embedding matrix through an initializer
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,      # WE DO NOT TRAIN IT!
    mask_zero=True,
)

# Given that our network is initialized randomly, the massive changes it undergoes at the beginning
# of training would certainly affect/damage the representations in our embedding matrix
# (same scenario as with pretrained ConvNets)

In [23]:
# A model that uses a pretrained Embedding layer
keras.utils.clear_session()

inputs = keras.Input(shape=(None,), dtype="int64")
# ↓ our embedding layer --------------------------------------------
embedded = embedding_layer(inputs)
# ---------------------------------------------------- passed here ↓
x = keras.layers.Bidirectional(keras.layers.LSTM(32))(embedded)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# model.summary()

In [24]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODELS_DIR / "glove_embeddings_sequence_model.keras"),
        save_best_only=True
    )
]
model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=0,
)

I0000 00:00:1772643918.377721 10537601 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [25]:
print(f"Test acc: {model.evaluate(sequence_test_ds, verbose=0)[1]:.3f}")

Test acc: 0.864
